---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 1.3: Evaluation Refinement (phase 1)

**Scenario:** You're an AI engineer at ACME Manufacturing. The company is interested in creating a high quality MLflow assistant for developers and ML Engineers in particular. Initial testing has kicked off and you are about to meet with Sam to collect initial feedback to refine your quality checks.

**Objective:** Translate feedback into evaluations and scorers that can be used to refine the agent. Refine aspects of the agent to meet updated evaluation criteria.

## 💬 Product Team Check-in #1

---

> **Sam (Product):** "Hey — the beta testers in ML Engineering have been active. Two pieces of feedback worth acting on. First, the tone feels a bit robotic. Users said it feels like they're reading documentation rather than talking to a helpful assistant."
>
> **You:** "Easy fix — that's a system prompt change plus a `Guidelines` scorer to catch it. I'll update the prompt to be more conversational and add an eval case or two that would fail with the current tone."
>
> **Sam:** "Great. Second thing — users are asking non-MLflow questions and the agent is just... answering them. Someone asked for a pasta recipe and it gave them one."
>
> **You:** "That's another possible system prompt guardrail. I'll add an explicit instruction to stay on topic and add a `Guidelines` scorer for it. I'll also add an eval case with an off-topic question so we can catch regressions if the guardrail ever slips."

---

Let's translate that into some evaluations.

## 1 - Environment Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import mlflow

load_dotenv()

openai_client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-2.5-flash-lite"
JUDGE_MODEL = "gemini:/gemini-3.1-flash-lite-preview"
EXPERIMENT_NAME = os.environ.get("EXPERIMENT_1_NAME", "mlflow-agent-1")

if not os.getenv("MLFLOW_TRACKING_URI"):
    mlflow.set_tracking_uri("http://127.0.0.1:5000")

experiment = mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.openai.autolog()

# Same as notebook 2
SYSTEM_PROMPT = """You are an MLflow assistant."""

def mlflow_agent(question: str) -> str:
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        temperature=0,
        max_completion_tokens=512,
    )
    return response.choices[0].message.content

print("Autologging enabled. Open the MLflow UI to follow along:")
print("  mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000")
print("If previous instance is running, run in terminal: kill -9 $(lsof -t -i:5000)")

Autologging enabled. Open the MLflow UI to follow along:
  mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
If previous instance is running, run in terminal: kill -9 $(lsof -t -i:5000)


## Changing Judge Model

Using the same LLM as both agent and judge creates correlated blind spots — the judge shares the agent's biases and failure modes, so it's more likely to rate flawed outputs as correct.

Common practice to choose a 

In [64]:
JUDGE_MODEL = "gemini:/gemini-3.1-flash-lite-preview"

# Evaluation Datasets

## The Problem with Inline Datasets

**Scaling issues:**
- Adding a new example means editing notebook code and re-running
- Creating via separate .py file can get long
- No way to tell who added which example, or when
- If two people add the same question, you get duplicates
- Deleting a bad example means finding it in a long list and hoping you don't break the format
- The dataset lives in notebook state — restart the kernel and it's gone until you re-run

Let's fix all of this by creating a **persistent evaluation dataset** with `mlflow.genai.datasets`. 


Evaluation datasets have the following features:


- **Persistence** — examples stored in a database, not scattered across notebook cells
- **CRUD operations** — add, update, or remove examples without re-running everything
- **Deduplication** — same inputs automatically merge instead of creating duplicates
- **Provenance** — know where each example came from (hand-written, from traces, from docs)
- **Tagging** — organize examples by category, priority, or sprint
- **Versioning** — the dataset has a content hash so you know when it changed

`create_dataset()` creates a new named dataset stored in your tracking server's SQL backend. It's attached to one or more experiments and can be tagged for organization.

**Requirements:** Your tracking server must use a SQL backend (SQLite, PostgreSQL, MySQL), which is the norm going forward. The `sqlite:///mlflow.db` we've been using works perfectly.


In [ ]:
from mlflow.genai.datasets import create_dataset

full_eval_data = create_dataset(
    name="mlflow-agent-eval",
    tags={
        "experiment": EXPERIMENT_NAME,
        "domain": "mlflow",
        "status": "active",
    },
)
full_eval_data.has_records()

In [56]:
type(full_eval_data)

mlflow.genai.datasets.evaluation_dataset.EvaluationDataset

WARNING: [Documentation issue](https://github.com/mlflow/mlflow/issues/22712)

In [16]:
#Retrieve the dataset later
from mlflow.genai.datasets import get_dataset
retrieved_dataset = get_dataset(name="mlflow-agent-eval")
# If multiple datasets have the same name, you can also retrieve by ID:
# retrieved_dataset = get_dataset(dataset_id="your-dataset-id")

### Add evaluation examples from previous notebok

`merge_records()` adds examples to the dataset. It's called "merge" because if you add an example with the same `inputs` as an existing record, it **updates** the existing record rather than creating a duplicate. This is based on an input hash — if the `inputs` dict is identical, it's the same record.


In [49]:
eval_dataset_v3 = [
    {
        "inputs": {"question": "How do I log a metric in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.log_metric", "key", "value", "step"],
        },
    },
    {
        "inputs": {"question": "How do I set up autologging for a scikit-learn model?"},
        "expectations": {
            "expected_response": "To enable autologging for scikit-learn in MLflow, call mlflow.sklearn.autolog() before your training code. This automatically logs parameters, metrics, and the trained model to your active run.",
        },
    },
    {
        "inputs": {"question": "How do I organize my runs in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.set_experiment", "experiment", "run"],
        },
    },
    {
        "inputs": {"question": "How do I log a model artifact in MLflow?"},
        "expectations": {
            "expected_response": "You can log a model artifact using mlflow.log_artifact() for individual files, or framework-specific methods like mlflow.sklearn.log_model() which saves the model along with its dependencies and a model signature.",
        },
    },
    {
        "inputs": {"question": "How do I compare runs in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.search_runs", "experiment_id", "metrics"],
        },
    },
]
print(f"Eval set size: {len(eval_dataset_v3)} examples")
#MLflow v3.11.1 bug: print(f"Evaluation data set size before merging: {len(full_eval_data.to_df())} records")
full_eval_data.merge_records(eval_dataset_v3)
print(f"Evaluation data set size after merging: {len(full_eval_data.to_df())} records")

Eval set size: 5 examples
Evaluation data set size after merging: 5 records


### Running eval in CI

```python
# This is all your CI script needs — the dataset is already populated
dataset = get_dataset(name="recipe-assistant-eval")
results = mlflow.genai.evaluate(data=dataset, predict_fn=recipe_agent, scorers=[...])
assert results.metrics["Correctness/percentage"] >= 0.8
```

### Growing from traces

```python
# Pull interesting production traces into your eval set
traces = mlflow.search_traces(
    filter_string="tags.user_feedback = 'negative'",
    max_results=20,
    return_type="list",
)
dataset.merge_records(traces)
```

# Problem Validation

## Introducing Guidelines Scorers

There are two ways to evaluate responses against custom rules in plain English. Which one you use depends on whether the guideline is **global** (same rule for every row) or **per-row** (different rules for different questions).

---

### Option A: `Guidelines` — Global rules passed to the scorer

Use this when the same rule applies to every example in your dataset. The scorer itself holds the rule.

**Required parameters:**
- **`name`** — a short identifier for the guideline
- **`guidelines`** — the rule the response must follow

```python
from mlflow.genai.scorers import Guidelines

# Applied to every row in the dataset
Guidelines(
    name="stays_on_topic",
    guidelines="Responses must only address MLflow-related questions.",
)
```

---

### Option B: `ExpectationGuidelines` — Per-row rules in the dataset

Use this when different questions need different rules. The guidelines live in each row's `expectations.guidelines` list, and the scorer reads them automatically.

```python
from mlflow.genai.scorers import ExpectationGuidelines

# No name or guidelines on the scorer — it reads from each row
ExpectationGuidelines()
```

The dataset provides the rules:

```python
{
    "inputs": {"question": "Can you give me a pasta recipe?"},
    "expectations": {
        "guidelines": [
            "Response must not provide non-MLflow content",
            "Response should redirect the user to ask an MLflow question",
        ],
    },
}
```

---

**We'll use `ExpectationGuidelines` here** since our two validation examples (tone and off-topic) each need their own specific rules.

In [62]:
validation_dataset_w_guidelines = [
    # Robotic tone — the bare "You are an MLflow assistant" prompt
    # tends to produce dry, documentation-style responses
    {
        "inputs": {"question": "What is MLflow tracing and why would I use it?"},
        "expectations": {
            "guidelines": [
                "Response should feel like advice from a knowledgeable colleague, "
                "not a dry enumeration of features or a copy-pasted changelog",
                "Response should explain the practical benefit to the user, "
                "not just list what the feature does",
            ],
        },
    },
    # Off-topic — the bare LLM will happily answer this
    {
        "inputs": {"question": "Can you give me a good pasta recipe?"},
        "expectations": {
            "guidelines": [
                "Response must not provide a pasta recipe or any non-MLflow content",
                "Response should politely redirect the user to ask an MLflow-related question",
            ],
        },
    },
]

In [ ]:
# Evaluate
from mlflow.genai import evaluate
from mlflow.genai.scorers import ExpectationsGuidelines

validation_results = evaluate(
    data=validation_dataset_w_guidelines,
    predict_fn=mlflow_agent,
    scorers=[ExpectationsGuidelines(model=JUDGE_MODEL)],
)

2026/04/22 09:31:10 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/04/22 09:31:10 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 2/2 [Elapsed: 00:09, Remaining: 00:00] [predict_fn: 66%, scorers: 34%]


In [53]:
validation_dataset_w_o_guidelines = [
    # Robotic tone — the bare "You are an MLflow assistant" prompt
    # tends to produce dry, documentation-style responses
    {
        "inputs": {"question": "What is MLflow tracing and why would I use it?"},
        "expectations": {},
    },
    # Off-topic — the bare LLM will happily answer this
    {
        "inputs": {"question": "Can you give me a good pasta recipe?"},
        "expectations": {},
    },
]

In [ ]:
# Evaluate
from mlflow.genai.scorers import Guidelines

validation_results = evaluate(
    data=validation_dataset_w_o_guidelines,
    predict_fn=mlflow_agent,
    scorers=[Guidelines(
        name="Non-robotic tone",
        guidelines="""Response should feel like advice from a knowledgeable colleague, "
                "not a dry enumeration of features or a copy-pasted changelog",
                "Response should explain the practical benefit to the user, "
                "not just list what the feature does""",
        model=JUDGE_MODEL),
        Guidelines(
        name="On Topic",
        guidelines="""Response should be related to MLflow context. 
        If question if off topic, response should politely redirect the user to ask an MLflow-related question""",
        model=JUDGE_MODEL),
        ]
)

## Add specific examples to dataset

In [55]:
full_eval_data.merge_records(validation_dataset_w_guidelines)
print(f"Number of records: {len(full_eval_data.to_df())}")

Number of records: 7


In [ ]:
# Add a trace to the mlflow dataset via UI
print(mlflow_agent("How do I search MLflow runs?"))

In [ ]:
# Run evaluation on evaluation dataset
#retrieved_dataset = get_dataset(name="mlflow-agent-eval")
from training.agents.scorers import MLFLOW_SCORERS
results = mlflow.genai.evaluate(
    data=full_eval_data,
    predict_fn=mlflow_agent,
    scorers=MLFLOW_SCORERS,
)


# Prompt Registration

---
## The Problem: Prompts as Strings

Our system prompt in its current form:

```python
SYSTEM_PROMPT = """You are an MLflow assistant."""

# After some feedback
SYSTEM_PROMPT_V2 = """You are a helpful MLflow assistant...Always include code examples and version context."""
```

Questions this process can't answer:
- Which prompt version was running when User #347 got a wrong API recommendation?
- Did changing "be practical" to "include code examples" improve Correctness scores?
- Can we roll back to last week's prompt without a code deploy?

The prompt registry makes all of this much more streamlined.

In [ ]:
SYSTEM_PROMPT = """You are an MLflow assistant."""

---
## 2 — Registering the System Prompt

`mlflow.genai.register_prompt()` creates a new prompt (or a new version of an existing one) in the registry. The prompt is stored in the tracking server's database and is immediately available to any notebook, script, or service that can reach the server.

Let's register the original system prompt from the first notebook — the bare-bones version before any of Sam's feedback.

In [ ]:
prompt_v1 = mlflow.genai.register_prompt(
    name="mlflow-agent-system",
    template=(SYSTEM_PROMPT),
    commit_message="Initial system prompt — bare PoC version",
    tags={
        "author": "ai-team",
        "agent": "mlflow-agent",
        "stage": "dev",
    },
)

print(f"Prompt: {prompt_v1.name}")
print(f"Version: {prompt_v1.version}")
print(f"Template: {prompt_v1.template[:80]}...")

In [ ]:
eval_dataset = [
    # API specificity — requires a real function signature, not a vague description
    {
        "inputs": {"question": "How do I run an evaluation in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.genai.evaluate", "data", "scorers"],
            "guidelines": [
                "Response must include a concrete code example, not just a description of what to do",
                "Response must reference specific parameter names, not just say 'pass in your data'",
            ],
        },
    },
    # Version clarity — requires distinguishing 3.0 from pre-3.0
    {
        "inputs": {"question": "How do I add a custom scorer to my evaluation?"},
        "expectations": {
            "expected_facts": ["@scorer", "mlflow.genai"],
            "guidelines": [
                "Response must indicate this API is specific to MLflow 3.0",
                "Response must not reference mlflow.evaluate() which is the pre-3.0 pattern",
            ],
        },
    },
    # Tone — conversational vs. changelog
    {
        "inputs": {"question": "What is MLflow tracing and why would I use it?"},
        "expectations": {
            "guidelines": [
                "Response should be conversational and helpful, not a dry enumeration of features",
                "Response should explain the practical benefit to the user, not just what the feature does",
            ],
        },
    },
    # Combines all three — specificity, version clarity, and tone
    {
        "inputs": {"question": "How do I trace a custom function in MLflow 3.0?"},
        "expectations": {
            "expected_facts": ["@mlflow.trace", "SpanType"],
            "guidelines": [
                "Response must include a code example showing the decorator in use",
                "Response must clarify this is a MLflow 3.0 feature",
                "Response should be conversational and explain when you would want to do this",
            ],
        },
    },
]
print(f"Eval set size: {len(eval_dataset)} examples")

In [ ]:
eval_dataset_v4 = [
    {
        "inputs": {"question": "Can you give me a good pasta recipe?"},
        "expectations": {
            "guidelines": [
                "Response must not provide a pasta recipe or any cooking-related content",
            ],
        },
    },
    {
        "inputs": {"question": "Is there a seahorse emoji?"},
        "expectations": {
            "guidelines": [
                "Response must not answer questions that aren't related to MLflow",
            ],
        },
    },
]
print(f"Eval set size: {len(eval_dataset_v4)} examples")

---

## 3 - Update System Prompt

In [ ]:
SYSTEM_PROMPT_V2 = """You are a helpful MLflow assistant.
Be practical and conversational — like a knowledgeable colleague, not a changelog.
Do not answer questions that aren't related to MLflow."""

In [ ]:
prompt_v2 = mlflow.genai.register_prompt(
    name="mlflow-agent-system",
    template=(SYSTEM_PROMPT_V2),
    commit_message="Added code examples, version context, and conversational tone per Sam's feedback",
    tags={
        "author": "ai-team",
        "agent": "mlflow-agent",
    },
)

print(f"Prompt: {prompt_v2.name}")
print(f"Version: {prompt_v2.version}")
print(f"Template: {prompt_v2.template[:80]}...")

In [58]:
mlflow.genai.load_prompt("prompts:/mlflow-agent-system/2")

MlflowException: RESOURCE_DOES_NOT_EXIST: Prompt with name=mlflow-agent-system not found

# Set Prompt Alias

A prompt alias in MLflow is a mutable, named reference (like "production" or "staging") that points to a specific prompt version in the Prompt Registry. To promote a new prompt version to production, assign the "production" alias to the desired version using `mlflow.genai.set_prompt_alias()`. This allows your applications to reference the alias (e.g., prompts:/my_prompt@production) instead of a hard-coded version, enabling seamless updates without redeploying code.

In [59]:
mlflow.genai.set_prompt_alias(name="mlflow-agent-system", version=2, alias="prod")

MlflowException: Prompt 'mlflow-agent-system' version 2 does not exist.

In [ ]:
# Update the predict_fn to use the registered prompt
def mlflow_agent(question: str) -> str:
    """MLflow assistant with updated prompt."""
    SYSTEM_PROMPT = mlflow.genai.load_prompt("prompts:/mlflow-agent-system@prod")
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        temperature=0.1,
        max_completion_tokens=512,
    )
    return response.choices[0].message.content

In [ ]:
# Evaluate
results = mlflow.genai.evaluate(
    data=full_eval_data,
    predict_fn=mlflow_agent,
    scorers=MLFLOW_SCORERS,
)

---
## 5 — Custom Scorers with `@scorer`

`Guidelines` scorers use an LLM judge, which makes them flexible but slow and non-deterministic compared to standard metrics. For rules that can be expressed as code, a `@scorer` function is faster, cheaper, and completely predictable.

Your function receives:
- `outputs` — the string returned by `predict_fn`
- `inputs` — the original input dict (keyword arg)
- `expectations` — the ground truth dict (keyword arg)

Return `bool`, `int`, `float`, or a `Feedback` object with a rationale.

In [ ]:
import re
from mlflow.genai.scorers import scorer, Feedback, Correctness, RelevanceToQuery, Guidelines


@scorer
def has_code_block(outputs: str, **kwargs) -> bool:
    """
    Checks that the response includes at least one code-like snippet.
    Matches patterns like: 'mlflow.something(', 'import mlflow', '@mlflow.trace'.
    This is a fast, deterministic check — no LLM call needed.
    """
    code_pattern = re.compile(
        r'(mlflow\.\w+\(|import mlflow|@mlflow\.\w+|'
        r'from mlflow|\.log_\w+|\.evaluate\(|\.autolog\()',
        re.IGNORECASE
    )
    return bool(code_pattern.search(outputs))


@scorer
def not_overwhelming(outputs: str, **kwargs) -> Feedback:
    """
    Checks the response isn't too long. Users want focused answers, not essays.
    Returns a Feedback object so we can see the character count in the UI.
    """
    char_count = len(outputs)
    passes = char_count <= 2000
    return Feedback(
        value=passes,
        rationale=f"{char_count} characters — {'within' if passes else 'over'} the 2000-char limit",
    )


results_v4 = mlflow.genai.evaluate(
    data=eval_dataset_v2,
    predict_fn=mlflow_agent,
    scorers=[
        Correctness(),
        RelevanceToQuery(),
        Guidelines(
            name="includes_code_examples",
            guidelines=(
                "When explaining an MLflow API or feature, the response must include "
                "at least one concrete code snippet with real function names and parameter names."
            ),
        ),
        Guidelines(
            name="includes_version_context",
            guidelines=(
                "When referencing MLflow APIs, the response should mention which version "
                "of MLflow the API applies to or was introduced in."
            ),
        ),
        has_code_block,         # fast deterministic check
        not_overwhelming,       # fast deterministic check with rationale
    ],
)

print("=== Run 3: Custom scorers added ===")
print(results_v4.metrics)

# Update to Product Team